# US Grid Congestion and Curtailment Analysis

**Question:** Across the entire US grid, where is transmission congestion and renewable curtailment costing the most wasted clean energy, what structural factors explain why some regions are worse than others, and where would new transmission or battery investment pay off fastest?

**Data sources:**
- EIA-930 Hourly Electric Grid Monitor (eia.gov) -- national hourly demand/generation/interchange by balancing authority, 2022-2024
- EPA eGRID 2023 (epa.gov) -- emissions intensity and generation mix by balancing authority
- HIFLD Transmission Lines (data.gov / ArcGIS) -- US high-voltage transmission network
- LBNL Queued Up 2026 (emp.lbl.gov) -- full US interconnection queue across all ISOs through end of 2025

**Methods:** descriptive aggregation, GIS mapping, k-means clustering, time-series decomposition, SARIMA forecasting, isolation forest anomaly detection, Kruskal-Wallis cross-region statistical testing, battery arbitrage simulation

## Section 0: Imports and Setup

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from folium.plugins import MarkerCluster
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

BASE_DIR   = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR   = os.path.join(BASE_DIR, 'data')
EIA930_DIR = os.path.join(DATA_DIR, 'eia930')
EGRID_DIR  = os.path.join(DATA_DIR, 'egrid')
HIFLD_DIR  = os.path.join(DATA_DIR, 'hifld')
LBNL_DIR   = os.path.join(DATA_DIR, 'lbnl_queue')

print('Data directory:', DATA_DIR)
print('All imports OK.')

## Section 1: Load and Clean EIA-930 Data

EIA-930 gives us hourly readings per balancing authority (BA) -- the regional grid operators that cover the US.
Each row is one BA at one hour. Key columns:
- `gen_solar_mw` / `gen_wind_mw` -- how much renewable power was produced
- `demand_mw` -- how much power was consumed in the region
- `interchange_mw` -- net flow into/out of the region (positive = importing)

We use the Adjusted versions of each column -- EIA fills in gaps and corrects outliers in these, making them more reliable than the raw reported values.

Six files cover 2022-2024 (six months each). Concatenating them gives a 3-year hourly panel -- long enough to detect seasonal patterns and multi-year trends.

In [ ]:
# load and concatenate all six EIA-930 CSV files
# each file is ~40MB, so this takes 30-60 seconds
eia_files = sorted([
    os.path.join(EIA930_DIR, f)
    for f in os.listdir(EIA930_DIR) if f.endswith('.csv')
])
print(f'Found {len(eia_files)} EIA-930 files:')
for f in eia_files:
    print(' ', os.path.basename(f))

df_raw = pd.concat(
    [pd.read_csv(f, low_memory=False) for f in eia_files],
    ignore_index=True
)
print(f'\nTotal rows loaded: {len(df_raw):,}')

In [ ]:
# rename the long EIA column names to short ones we can type easily
# we use only the (Adjusted) versions of demand/generation/interchange --
# EIA fills missing BA reports using interpolation in these columns, so they
# have fewer gaps than the raw reported values
col_map = {
    'Balancing Authority':                                               'ba',
    'UTC Time at End of Hour':                                           'utc_time',
    'Demand (MW) (Adjusted)':                                            'demand_mw',
    'Net Generation (MW) (Adjusted)':                                    'gen_total_mw',
    'Total Interchange (MW) (Adjusted)':                                 'interchange_mw',
    'Net Generation (MW) from Solar (Adjusted)':                         'gen_solar_mw',
    'Net Generation (MW) from Wind (Adjusted)':                          'gen_wind_mw',
    'Net Generation (MW) from Hydropower and Pumped Storage (Adjusted)': 'gen_hydro_mw',
    'Net Generation (MW) from Natural Gas (Adjusted)':                   'gen_gas_mw',
    'Net Generation (MW) from Coal (Adjusted)':                          'gen_coal_mw',
    'Net Generation (MW) from Nuclear (Adjusted)':                       'gen_nuclear_mw',
    'Region':                                                            'region',
}
df = df_raw[list(col_map.keys())].rename(columns=col_map).copy()

# parse timestamp and extract useful time features
df['utc_time'] = pd.to_datetime(df['utc_time'], format='mixed')
df['year']  = df['utc_time'].dt.year
df['month'] = df['utc_time'].dt.month
df['hour']  = df['utc_time'].dt.hour
df['date']  = df['utc_time'].dt.date

# force all generation/demand columns to numeric -- some rows may have text placeholders
for col in ['demand_mw','gen_total_mw','interchange_mw','gen_solar_mw','gen_wind_mw',
            'gen_hydro_mw','gen_gas_mw','gen_coal_mw','gen_nuclear_mw']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# compute two core derived columns used throughout the analysis:
# renewable_mw: total solar + wind output per hour per BA
# renewable_share: what fraction of that BA's total generation came from renewables
# gen_demand_surplus: generation minus demand -- positive = the BA is producing more than it needs
#   and must export; a large or volatile surplus is our main congestion proxy since we
#   do not have node-level LMP prices in this dataset
df['renewable_mw']       = df['gen_solar_mw'].fillna(0) + df['gen_wind_mw'].fillna(0)
df['renewable_share']    = df['renewable_mw'] / df['gen_total_mw'].replace(0, np.nan)
df['gen_demand_surplus'] = df['gen_total_mw'] - df['demand_mw']

print(f'Cleaned shape: {df.shape}')
print(f'Date range: {df["utc_time"].min()} to {df["utc_time"].max()}')
print(f'Unique BAs: {df["ba"].nunique()} | Regions: {list(df["region"].unique())}')
df.head(3)

## Section 2: Load eGRID and LBNL Queue Data

In [ ]:
# load eGRID balancing authority sheet (BA23 tab)
# eGRID gives us 2023 annual totals per BA: total generation, CO2 emissions, nameplate capacity
# we use CO2 intensity as a structural control in the clustering step --
# it captures whether a BA is fossil-heavy or already clean, which is independent of
# our congestion proxy and adds useful explanatory variation
#
# header=0 picks the first Excel row which has human-readable column names
# skiprows=[1] drops the second row which contains short codes (BACODE, BANGENAN, etc.)
# that would otherwise become data rows
egrid_ba = pd.read_excel(
    os.path.join(EGRID_DIR, 'egrid2023_data_rev2.xlsx'),
    sheet_name='BA23',
    header=0,
    skiprows=[1]
)

egrid_ba = egrid_ba[[
    'Balancing Authority Code',
    'BA nameplate capacity (MW)',
    'BA annual net generation (MWh)',
    'BA annual CO2 emissions (tons)'
]].rename(columns={
    'Balancing Authority Code':       'ba',
    'BA nameplate capacity (MW)':     'nameplate_cap_mw',
    'BA annual net generation (MWh)': 'annual_gen_mwh',
    'BA annual CO2 emissions (tons)': 'annual_co2_tons'
})
for col in ['nameplate_cap_mw','annual_gen_mwh','annual_co2_tons']:
    egrid_ba[col] = pd.to_numeric(egrid_ba[col], errors='coerce')
egrid_ba['co2_intensity'] = egrid_ba['annual_co2_tons'] / egrid_ba['annual_gen_mwh'].replace(0, np.nan)
egrid_ba = egrid_ba.dropna(subset=['ba'])

print(f'eGRID loaded: {egrid_ba.shape[0]} balancing authorities')
egrid_ba.head(3)

In [ ]:
# load the LBNL Queued Up interconnection queue dataset
# sheet '03. Complete Queue Data' has one row per proposed power project
# header=1 because row 0 is a 'RETURN TO CONTENTS' navigation cell in the Excel file
#
# key columns we care about:
#   q_status  -- 'withdrawn' (gave up), 'operational' (built), 'active' (still waiting)
#   q_date    -- when the project entered the queue
#   on_date   -- when it went live (only for operational)
#   wd_date   -- when it withdrew (only for withdrawn)
#   region    -- which ISO/RTO (CAISO, PJM, MISO, ERCOT, etc.)
lbnl_raw = pd.read_excel(
    os.path.join(LBNL_DIR, 'queued_up_2026.xlsx'),
    sheet_name='03. Complete Queue Data',
    header=1
)

# select the columns we need, using a flexible approach that only keeps columns that exist
# (column names vary slightly across LBNL edition years)
desired_cols = ['q_id','q_status','q_date','on_date','wd_date','county','state','region','project_name']
extra_cols   = [c for c in lbnl_raw.columns if any(kw in c.lower() for kw in ['cap','mw','type','fuel','tech','resource'])]
keep_cols    = [c for c in desired_cols + extra_cols if c in lbnl_raw.columns]
lbnl = lbnl_raw[keep_cols].copy()

for dc in ['q_date','on_date','wd_date']:
    if dc in lbnl.columns:
        lbnl[dc] = pd.to_datetime(lbnl[dc], errors='coerce')

# queue_days: how long a project spent in the queue before being built or withdrawn
# we use whichever outcome date is available (on_date for built, wd_date for withdrawn)
lbnl['outcome_date'] = lbnl['on_date'].fillna(lbnl['wd_date'])
lbnl['queue_days']   = (lbnl['outcome_date'] - lbnl['q_date']).dt.days

print(f'LBNL queue loaded: {len(lbnl):,} projects')
print(f'Status breakdown:\n{lbnl["q_status"].value_counts()}')
lbnl.head(3)

## Section 3: Congestion Stress Ranking by Balancing Authority

We build a composite **congestion stress score** for each BA using three signals:
1. **Renewable share** -- high renewable penetration correlates with curtailment risk because solar/wind generate in geographic clusters that may not have enough transmission capacity to export all the power
2. **Surplus volatility** (std dev of generation-minus-demand) -- a BA that frequently swings between large surplus and large deficit is straining its transmission boundaries
3. **Heavy export hours** -- hours where surplus exceeds 50% of demand, the highest-risk window for curtailment

Each feature is z-scored so they contribute equally regardless of units, then averaged into one score.

In [ ]:
# aggregate EIA-930 from hourly rows to one summary row per BA
ba_profile = df.groupby('ba').agg(
    region              = ('region', 'first'),
    avg_demand_mw       = ('demand_mw', 'mean'),
    avg_gen_mw          = ('gen_total_mw', 'mean'),
    avg_renewable_mw    = ('renewable_mw', 'mean'),
    avg_renewable_share = ('renewable_share', 'mean'),
    std_surplus         = ('gen_demand_surplus', 'std'),
    avg_surplus         = ('gen_demand_surplus', 'mean'),
    avg_solar_mw        = ('gen_solar_mw', 'mean'),
    avg_wind_mw         = ('gen_wind_mw', 'mean'),
    hours_total         = ('utc_time', 'count'),
).reset_index()

# count hours where surplus exceeded 50% of local demand -- highest curtailment-risk hours
heavy_export = (
    df[df['gen_demand_surplus'] > df['demand_mw'] * 0.5]
    .groupby('ba').size()
    .reset_index(name='hours_heavy_export')
)
ba_profile = ba_profile.merge(heavy_export, on='ba', how='left')
ba_profile['heavy_export_pct'] = ba_profile['hours_heavy_export'] / ba_profile['hours_total']

# join eGRID structural data for later use in clustering
ba_profile = ba_profile.merge(egrid_ba, on='ba', how='left')

# build the composite congestion stress score
# z-score each feature so they are on the same scale, then average
stress_features = ['avg_renewable_share', 'std_surplus', 'heavy_export_pct']
stress_df = ba_profile[['ba', 'region'] + stress_features].dropna()
stress_scaled = StandardScaler().fit_transform(stress_df[stress_features])
stress_df = stress_df.copy()
stress_df['congestion_stress_score'] = stress_scaled.mean(axis=1)
ba_profile = ba_profile.merge(stress_df[['ba','congestion_stress_score']], on='ba', how='left')

top_stressed = (
    ba_profile[['ba','region','avg_renewable_share','std_surplus','heavy_export_pct','congestion_stress_score']]
    .dropna(subset=['congestion_stress_score'])
    .sort_values('congestion_stress_score', ascending=False)
    .head(20).reset_index(drop=True)
)
print('Top 20 most congestion-stressed balancing authorities:')
top_stressed

In [ ]:
# horizontal bar chart -- most stressed BAs at the top, colored red-to-green
fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(top_stressed)))
ax.barh(top_stressed['ba'][::-1], top_stressed['congestion_stress_score'][::-1], color=colors)
ax.set_xlabel('Congestion Stress Score (z-score composite)')
ax.set_title('Top 20 Most Congestion-Stressed Balancing Authorities (2022-2024)')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'congestion_stress_ranking.png'), dpi=150)
plt.show()

## Section 4: GIS Mapping

We overlay BA congestion scores on the actual US transmission network (HIFLD shapefile) and US state boundaries.
The goal: visually confirm that high-stress BAs sit on known bottleneck corridors.

Two outputs:
1. Static matplotlib map (save as PNG for a portfolio PDF)
2. Interactive folium map (open as HTML in a browser -- hover each dot for details)

In [ ]:
# load HIFLD transmission lines -- 52k line segments representing the US high-voltage grid
print('Loading transmission lines (~30 seconds)...')
transmission = gpd.read_file(os.path.join(HIFLD_DIR, 'transmission_lines.geojson')).to_crs(epsg=4326)
print(f'Transmission lines: {len(transmission):,} segments')

# load US state boundaries downloaded alongside the HIFLD data
us_states = gpd.read_file(os.path.join(HIFLD_DIR, 'us_states.geojson'))
print(f'US states: {len(us_states)} polygons')

In [ ]:
# approximate centroid coordinates for each balancing authority
# EIA-930 has no built-in coordinates, so we provide a lookup table
# these are approximate geographic centers of each BA's service territory,
# cross-referenced against EIA's own BA service territory map
ba_coords = {
    'CISO':(36.7,-119.7), 'ERCO':(31.0,-100.0), 'PJM':(40.0,-77.0),  'MISO':(41.5,-88.0),
    'SPP':(38.0,-97.5),   'ISNE':(42.3,-71.8),   'NYIS':(42.9,-74.6), 'SWPP':(38.0,-97.5),
    'BPAT':(45.5,-122.0), 'PACW':(43.0,-120.0),  'PACE':(40.5,-111.0),'WACM':(39.0,-105.0),
    'PSCO':(39.7,-104.9), 'SRP':(33.4,-112.0),   'AZPS':(33.4,-112.3),'WALC':(34.5,-114.0),
    'NEVP':(39.5,-117.0), 'LDWP':(34.0,-118.2),  'IID':(33.0,-115.5), 'TIDC':(37.6,-121.0),
    'BANC':(38.5,-121.5), 'AVA':(47.7,-117.4),   'CHPD':(47.9,-119.9),'DOPD':(47.7,-119.7),
    'GCPD':(47.0,-120.5), 'PSEI':(47.9,-122.1),  'TPWR':(47.2,-122.5),'SCL':(47.6,-122.3),
    'IPCO':(43.5,-116.2), 'NWMT':(46.5,-110.0),  'GWA':(46.0,-112.5), 'HGMA':(33.0,-112.0),
    'TEPC':(32.2,-110.9), 'EPE':(31.8,-106.4),   'PNM':(35.1,-106.6), 'WAPA':(39.0,-105.0),
    'AECI':(37.5,-91.8),  'LGEE':(38.2,-85.7),   'TVA':(35.5,-86.5),  'SOCO':(33.0,-84.0),
    'FPL':(26.7,-80.3),   'FPC':(28.8,-82.5),    'TEC':(27.9,-82.5),  'JEA':(30.3,-81.7),
    'SEC':(28.5,-81.4),   'NSB':(29.0,-80.9),    'HST':(25.8,-80.2),  'SEPA':(32.5,-83.6),
    'SCEG':(34.0,-81.0),  'SC':(34.0,-81.0),     'DUK':(35.2,-80.8),  'CPLE':(35.8,-79.0),
    'CPLW':(35.0,-81.5),  'DEF':(28.7,-81.5),    'DOM':(37.5,-77.5),  'OVEC':(38.9,-83.4),
    'AEC':(30.7,-88.0),   'EEI':(38.6,-87.5),    'YAD':(35.4,-80.0),  'GVL':(29.6,-82.3),
}

stress_geo = stress_df.copy()
stress_geo['lat'] = stress_geo['ba'].map(lambda x: ba_coords.get(x, (None,None))[0])
stress_geo['lon'] = stress_geo['ba'].map(lambda x: ba_coords.get(x, (None,None))[1])
stress_geo = stress_geo.dropna(subset=['lat','lon'])

gdf_ba = gpd.GeoDataFrame(
    stress_geo,
    geometry=gpd.points_from_xy(stress_geo['lon'], stress_geo['lat']),
    crs='EPSG:4326'
)

# clip all layers to continental US bounding box
conus_bbox = (-130, 22, -60, 52)
transmission_conus = transmission.cx[conus_bbox[0]:conus_bbox[2], conus_bbox[1]:conus_bbox[3]]
us_states_conus    = us_states.cx[conus_bbox[0]:conus_bbox[2], conus_bbox[1]:conus_bbox[3]]
gdf_conus          = gdf_ba.cx[conus_bbox[0]:conus_bbox[2], conus_bbox[1]:conus_bbox[3]]

print(f'BAs with coordinates mapped: {len(gdf_conus)}')

In [ ]:
# static matplotlib map
# layer order: state outlines (bottom) -> transmission lines -> BA stress dots -> labels
# circle size = renewable share (bigger = more renewables)
# circle color = stress score (red = most stressed, green = least)
fig, ax = plt.subplots(figsize=(18, 10))

us_states_conus.plot(ax=ax, color='#f5f5f5', edgecolor='#aaaaaa', linewidth=0.5)
transmission_conus.plot(ax=ax, color='lightgray', linewidth=0.3, alpha=0.6)
gdf_conus.plot(
    ax=ax,
    column='congestion_stress_score',
    cmap='RdYlGn_r',
    markersize=gdf_conus['avg_renewable_share'].fillna(0).clip(0.01, 1) * 200 + 30,
    legend=True,
    legend_kwds={'label': 'Congestion Stress Score', 'shrink': 0.6},
    alpha=0.85, edgecolor='black', linewidth=0.5,
)
for _, row in gdf_conus.nlargest(10, 'congestion_stress_score').iterrows():
    ax.annotate(row['ba'], xy=(row.geometry.x, row.geometry.y),
                xytext=(5, 5), textcoords='offset points',
                fontsize=7, fontweight='bold', color='darkred')

ax.set_xlim(-130, -60); ax.set_ylim(22, 52)
ax.set_title(
    'US Grid Congestion Stress by Balancing Authority (2022-2024)\n'
    'Circle size = renewable share | Color = stress score (red = most stressed)',
    fontsize=13
)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'congestion_map_static.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Static map saved.')

In [ ]:
# interactive folium map -- open the saved HTML file in any browser
# each circle is one BA; hover over it to see name, stress score, renewable share
# transmission lines are shown as thin gray lines in the background
m = folium.Map(location=[38, -96], zoom_start=4, tiles='CartoDB positron')

# sample every 10th transmission line to keep the map file size manageable
folium.GeoJson(
    transmission_conus.iloc[::10].__geo_interface__,
    style_function=lambda x: {'color': '#cccccc', 'weight': 0.5, 'opacity': 0.5}
).add_to(m)

vmin = gdf_conus['congestion_stress_score'].min()
vmax = gdf_conus['congestion_stress_score'].max()

def stress_to_color(val):
    # map normalized stress score to a hex color: red (high stress) to green (low stress)
    if pd.isna(val): return '#aaaaaa'
    n = (val - vmin) / (vmax - vmin + 1e-9)
    return f'#{int(min(255, n*510)):02x}{int(min(255, (1-n)*510)):02x}00'

for _, row in gdf_conus.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=max(6, row['avg_renewable_share'] * 25 if not pd.isna(row['avg_renewable_share']) else 6),
        color='black', weight=0.5, fill=True,
        fill_color=stress_to_color(row['congestion_stress_score']),
        fill_opacity=0.8,
        tooltip=folium.Tooltip(
            f"<b>{row['ba']}</b> ({row['region']})<br>"
            f"Stress Score: {row['congestion_stress_score']:.2f}<br>"
            f"Renewable Share: {row['avg_renewable_share']:.1%}<br>"
            f"Heavy Export Hours: {row['heavy_export_pct']:.1%}"
        )
    ).add_to(m)

folium.LayerControl().add_to(m)
map_path = os.path.join(BASE_DIR, 'congestion_map_interactive.html')
m.save(map_path)
print(f'Interactive map saved: {map_path}')
print('Open congestion_map_interactive.html in your browser to explore.')

## Section 5: Clustering -- Bottleneck Archetypes

K-means clustering groups BAs by similarity across four features simultaneously:
renewable share, surplus volatility, heavy-export frequency, and CO2 intensity.

The goal is to find distinct *types* of grid stress -- not just rank BAs by a single score.
A region that is highly renewable and chronically exporting is a different problem than one
that is fossil-heavy with occasional extreme demand spikes.

We choose k using the silhouette score -- it measures how well-separated clusters are (1 = perfect, -1 = bad).

In [ ]:
cluster_features = ['avg_renewable_share', 'std_surplus', 'heavy_export_pct', 'co2_intensity']
cluster_df = ba_profile[['ba', 'region'] + cluster_features].dropna()
X = StandardScaler().fit_transform(cluster_df[cluster_features])

# test k from 2 to 8, pick the k with the highest silhouette score
sil_scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    sil_scores[k] = silhouette_score(X, km.fit_predict(X))
best_k = max(sil_scores, key=sil_scores.get)
print(f'Silhouette scores: {sil_scores}')
print(f'Best k: {best_k}')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(sil_scores.keys()), list(sil_scores.values()), 'o-')
ax.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
ax.set_xlabel('Number of clusters (k)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Choosing the number of bottleneck archetypes')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# fit final model with best_k and show centroid values to interpret each cluster
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_df = cluster_df.copy()
cluster_df['cluster'] = km_final.fit_predict(X)

centroids = cluster_df.groupby('cluster')[cluster_features].mean().round(3)
print('Cluster centroids (mean per feature, unscaled):')
print(centroids)
print('\nBAs per cluster:')
print(cluster_df.groupby('cluster')['ba'].apply(list))

In [ ]:
# assign human-readable names based on the centroid values above
# look at the centroids table and update these labels to reflect what you see
# typical patterns:
#   high renewable_share + high heavy_export_pct  --> 'Renewable Exporter (Congestion Risk)'
#   low renewable + high std_surplus              --> 'Volatile Fossil Grid'
#   low renewable + low std_surplus + high co2    --> 'Stable Baseload (Fossil)'
#   moderate renewable + low stress               --> 'Transitioning Grid'

archetype_labels = {i: f'Archetype {i}' for i in range(best_k)}
# update the dict above with your own names after reading the centroids, for example:
# archetype_labels = {0: 'Stable Baseload', 1: 'Renewable Exporter', ...}

cluster_df['archetype'] = cluster_df['cluster'].map(archetype_labels)
print('BAs per archetype:')
print(cluster_df['archetype'].value_counts())
cluster_df[['ba','region','cluster','archetype']].sort_values('cluster')

## Section 6: Time-Series Decomposition and Forecasting

We take the highest-stress BA and decompose its daily surplus time series into:
- **Trend**: is congestion getting worse over the 3-year period?
- **Seasonality**: which season is most stressed (summer heat? winter wind lulls?)?
- **Residual**: unexplained variation after removing trend and seasonality

Then we build a SARIMA model to forecast the next 12 months.
SARIMA = Seasonal ARIMA, an autoregressive model that accounts for both short-term
autocorrelation and annual seasonality.

In [ ]:
# pick the top-stressed BA for the deep dive
top_ba = top_stressed.iloc[0]['ba']
print(f'Deep-dive BA: {top_ba} (region: {top_stressed.iloc[0]["region"]})')

# aggregate hourly data to daily -- hourly is too noisy for decomposition,
# monthly loses the within-season detail we care about
ts_ba = (
    df[df['ba'] == top_ba]
    .groupby('date')
    .agg(daily_surplus=('gen_demand_surplus','mean'),
         daily_renewable_mw=('renewable_mw','mean'),
         daily_demand_mw=('demand_mw','mean'))
    .reset_index()
)
ts_ba['date'] = pd.to_datetime(ts_ba['date'])
ts_ba = ts_ba.set_index('date').sort_index()
print(f'Daily series: {len(ts_ba)} days')
ts_ba.head()

In [ ]:
# additive decomposition: observed = trend + seasonal + residual
# additive (not multiplicative) because the seasonal swing amplitude stays roughly
# constant over the period rather than growing proportionally with the trend level
# period=365 captures the annual solar/wind cycle
ts_clean = ts_ba['daily_surplus'].dropna()
decomp = seasonal_decompose(ts_clean, model='additive', period=365, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title=f'{top_ba}: Observed Daily Surplus (MW)')
decomp.trend.plot(ax=axes[1], title='Trend -- is stress growing over time?')
decomp.seasonal.plot(ax=axes[2], title='Seasonality -- which months are worst?')
decomp.resid.plot(ax=axes[3], title='Residual -- unexplained spikes')
for ax in axes:
    ax.set_xlabel('')
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
plt.suptitle(f'Seasonal Decomposition -- {top_ba}', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, f'decomposition_{top_ba}.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SARIMA forecast: aggregate to monthly first (daily SARIMA on 1000+ points is very slow)
# SARIMA(1,1,1)(1,1,1,12) means:
#   (1,1,1): AR lag 1, first-difference for stationarity, MA lag 1
#   (1,1,1,12): same structure repeated at the 12-month seasonal period
# this is a standard starting spec; a production model would tune via AIC/BIC grid search
ts_monthly = ts_clean.resample('ME').mean()

sarima_model = SARIMAX(
    ts_monthly,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
result = sarima_model.fit(disp=False)

forecast    = result.get_forecast(steps=12)
fc_mean     = forecast.predicted_mean
fc_ci       = forecast.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(14, 5))
ts_monthly.plot(ax=ax, label='Historical monthly surplus', color='steelblue')
fc_mean.plot(ax=ax, label='12-month forecast', color='darkorange', linestyle='--')
ax.fill_between(fc_ci.index, fc_ci.iloc[:,0], fc_ci.iloc[:,1],
                color='darkorange', alpha=0.2, label='95% confidence interval')
ax.set_title(f'{top_ba}: Monthly Grid Surplus Forecast (SARIMA)', fontsize=13)
ax.set_ylabel('Avg Daily Surplus (MW)')
ax.axhline(0, color='black', linewidth=0.6, linestyle='--')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, f'forecast_{top_ba}.png'), dpi=150, bbox_inches='tight')
plt.show()
print(result.summary())

## Section 7: Anomaly Detection -- Extreme Grid Events

Isolation Forest flags hours that are statistical outliers relative to a region's own normal range.
It works by randomly partitioning the feature space: anomalies sit in sparse regions and get
isolated with fewer splits than normal points, giving them a low anomaly score.

We run it per region (not globally) so that a Texas summer peak is compared against Texas norms,
not against the scale of a much larger region like PJM.

We then check whether anomaly frequency is increasing over time -- if so, the grid's stress
is worsening faster than the infrastructure is being upgraded.

In [ ]:
anomaly_features = ['demand_mw','gen_total_mw','renewable_mw','interchange_mw','gen_demand_surplus']
anomaly_results  = []

for region, group in df.groupby('region'):
    sub = group[anomaly_features].dropna()
    if len(sub) < 100:
        continue
    X_anom  = StandardScaler().fit_transform(sub)
    iso     = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
    preds   = iso.fit_predict(X_anom)
    sr      = group.loc[sub.index].copy()
    sr['anomaly_flag']  = (preds == -1).astype(int)
    sr['anomaly_score'] = iso.score_samples(X_anom)
    anomaly_results.append(sr)

df_anomaly = pd.concat(anomaly_results, ignore_index=True)
print(f'Anomalies flagged: {df_anomaly["anomaly_flag"].sum():,} of {len(df_anomaly):,} hours ({df_anomaly["anomaly_flag"].mean():.2%})')

In [ ]:
# zoom in on ERCOT (Texas) -- largest deregulated market, well-known for extreme events
# our data starts 2022 so the Feb 2021 Winter Storm Uri is outside range,
# but Summer 2023 set multiple grid emergency records and should surface clearly
erco = df_anomaly[df_anomaly['region'] == 'TEX'].copy()
normal  = erco[erco['anomaly_flag'] == 0]
anomaly = erco[erco['anomaly_flag'] == 1]

fig, ax = plt.subplots(figsize=(16, 5))
ax.scatter(normal['utc_time'],  normal['demand_mw'],  s=0.5, color='steelblue', alpha=0.3, label='Normal')
ax.scatter(anomaly['utc_time'], anomaly['demand_mw'], s=8,   color='red',       alpha=0.8, label='Anomaly')
ax.set_title('ERCOT (Texas): Demand with Anomalous Hours Flagged by Isolation Forest')
ax.set_ylabel('Demand (MW)'); ax.set_xlabel('Date'); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'anomaly_erco.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# is anomaly frequency increasing over time by region?
# rising anomaly rate = the grid's stress is outpacing infrastructure upgrades
anomaly_trend = (
    df_anomaly
    .groupby(['year','region'])
    .agg(anomaly_rate=('anomaly_flag','mean'))
    .reset_index()
)
fig, ax = plt.subplots(figsize=(12, 5))
for region, group in anomaly_trend.groupby('region'):
    ax.plot(group['year'], group['anomaly_rate'], marker='o', label=region)
ax.set_title('Annual Anomaly Rate by Region (2022-2024)')
ax.set_ylabel('Fraction of hours flagged as anomalous')
ax.set_xlabel('Year')
ax.legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'anomaly_trend_by_region.png'), dpi=150, bbox_inches='tight')
plt.show()

## Section 8: Cross-Region Statistical Comparison

We formally test whether congestion stress levels differ significantly across regions.
This answers: does regional market design or grid structure explain why some areas are
consistently more stressed than others?

We use the **Kruskal-Wallis test** -- the non-parametric equivalent of one-way ANOVA.
We choose non-parametric because the stress distribution is right-skewed (a few extreme
months dominate), so the normal-distribution assumption of ANOVA does not hold.

In [ ]:
# compute monthly surplus volatility (std dev) per BA per month -- our test variable
# this is more granular than the annual stress score and gives enough data points
# per region to make the hypothesis test meaningful
monthly_stress = (
    df.groupby(['ba','region','year','month'])
    .agg(monthly_surplus_std=('gen_demand_surplus','std'))
    .reset_index()
    .dropna()
)

region_groups = [g['monthly_surplus_std'].values for _, g in monthly_stress.groupby('region') if len(g) > 5]
region_names  = [n for n, g in monthly_stress.groupby('region') if len(g) > 5]

# H0: all regions have the same distribution of monthly stress
# reject H0 if p < 0.05
kw_stat, kw_pval = stats.kruskal(*region_groups)
print(f'Kruskal-Wallis H = {kw_stat:.2f}, p = {kw_pval:.2e}')
print('Regions differ significantly in congestion stress.' if kw_pval < 0.05 else 'No significant difference detected.')

In [ ]:
# box plot -- visual intuition behind the Kruskal-Wallis result
# each box shows the distribution of monthly stress for one region
fig, ax = plt.subplots(figsize=(14, 6))
region_data = [
    monthly_stress[monthly_stress['region']==r]['monthly_surplus_std'].dropna().values
    for r in region_names
]
ax.boxplot(region_data, labels=region_names, patch_artist=True,
           medianprops={'color':'red','linewidth':2})
ax.set_title(f'Monthly Congestion Stress by Region (2022-2024) -- Kruskal-Wallis p={kw_pval:.2e}')
ax.set_ylabel('Std Dev of Hourly Generation Surplus (MW)')
ax.set_xlabel('EIA Region')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'cross_region_stress_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## Section 9: Battery Arbitrage Value Simulation

For the top congested BAs we simulate a 100 MW / 4-hour battery (400 MWh total)
running a simple rule-based arbitrage:
- **Charge** when generation surplus is positive (grid has excess, curtailment risk)
- **Discharge** when surplus is negative (grid is tight, import needed)

The output -- MWh of curtailed surplus absorbed per year -- is converted to a dollar value
using a conservative $30/MWh curtailment avoidance estimate from NREL studies.
This is the "where would storage pay off fastest" answer that consulting clients pay for.

In [ ]:
BATTERY_MW           = 100   # power rating in MW
BATTERY_DURATION     = 4     # hours of storage at full power
BATTERY_MWH          = BATTERY_MW * BATTERY_DURATION  # energy capacity in MWh
EFFICIENCY           = 0.85  # round-trip efficiency: 85% of energy in comes back out
AVOIDED_COST_PER_MWH = 30    # USD/MWh -- conservative curtailment avoidance value


def simulate_battery_arbitrage(surplus_series):
    # simulate one full charge/discharge cycle over the time series
    # soc = state of charge in MWh, starts empty
    # total_absorbed = total MWh of surplus absorbed over the full period
    # we cap charge/discharge at the battery's power and energy limits each hour
    # efficiency penalty applies on discharge: you lose 15% of what you put in
    soc, total_absorbed = 0.0, 0.0
    for surplus in surplus_series:
        if pd.isna(surplus):
            continue
        if surplus > 0:
            charge = min(surplus, BATTERY_MW, BATTERY_MWH - soc)
            soc += charge
            total_absorbed += charge
        else:
            discharge = min(-surplus, BATTERY_MW, soc * EFFICIENCY)
            soc = max(soc - discharge / EFFICIENCY, 0)
    return total_absorbed


battery_results = []
for ba in top_stressed['ba'].tolist():
    series = df[df['ba'] == ba]['gen_demand_surplus'].values
    if len(series) < 100:
        continue
    annual_mwh = simulate_battery_arbitrage(series) / 3  # 3 years -> annualize
    battery_results.append({
        'ba':                  ba,
        'annual_mwh_absorbed': round(annual_mwh),
        'annual_value_usd':    round(annual_mwh * AVOIDED_COST_PER_MWH),
    })

battery_df = pd.DataFrame(battery_results).sort_values('annual_value_usd', ascending=False)
print(f'Battery sim results ({BATTERY_MW}MW / {BATTERY_DURATION}-hr):')
battery_df

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(
    battery_df['ba'],
    battery_df['annual_value_usd'] / 1e6,
    color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.7, len(battery_df)))
)
ax.set_title(
    f'Estimated Annual Battery Value by BA\n'
    f'({BATTERY_MW}MW / {BATTERY_DURATION}-hr @ ${AVOIDED_COST_PER_MWH}/MWh curtailment avoidance)'
)
ax.set_ylabel('Annual Value ($ millions)')
ax.set_xlabel('Balancing Authority')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'battery_arbitrage_value.png'), dpi=150, bbox_inches='tight')
plt.show()

## Section 10: Interconnection Queue Analysis

The LBNL Queued Up dataset shows where new clean energy projects are trying to connect to the grid.
Two questions:
1. Which regions have the highest project withdrawal rates? (= where the queue is killing projects)
2. Is the withdrawal rate getting worse over time? (= is the crisis deepening?)

Overlapping this with our congestion map makes the consulting narrative concrete:
the most congested regions are also the ones where new clean energy projects are failing
to get built -- confirming transmission as the bottleneck.

In [ ]:
# withdrawal rate by ISO/RTO region
# note: q_status values are 'withdrawn', 'operational', 'active', 'suspended'
# we use == 'withdrawn' (exact match) not startswith('wd') which would not match
queue_by_region = (
    lbnl
    .assign(withdrawn=lambda x: (x['q_status'] == 'withdrawn').astype(int))
    .groupby('region')
    .agg(
        total_projects  = ('q_id', 'count'),
        withdrawn_count = ('withdrawn', 'sum'),
        avg_queue_days  = ('queue_days', 'mean'),
    )
    .reset_index()
)
queue_by_region['withdrawal_rate'] = queue_by_region['withdrawn_count'] / queue_by_region['total_projects']
queue_by_region = queue_by_region.sort_values('withdrawal_rate', ascending=False)
print('Queue withdrawal rates by region:')
queue_by_region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
q = queue_by_region.dropna(subset=['withdrawal_rate']).head(15)

axes[0].barh(q['region'][::-1], q['withdrawal_rate'][::-1], color='salmon', edgecolor='black', linewidth=0.5)
axes[0].axvline(q['withdrawal_rate'].mean(), color='red', linestyle='--', label='Average')
axes[0].set_xlabel('Withdrawal Rate'); axes[0].set_title('Queue Withdrawal Rate by Region'); axes[0].legend()

q2 = queue_by_region.dropna(subset=['avg_queue_days']).head(15)
axes[1].barh(q2['region'][::-1], q2['avg_queue_days'][::-1] / 365, color='steelblue', edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Average Queue Duration (Years)'); axes[1].set_title('Avg Time in Queue by Region')

plt.suptitle('LBNL Queued Up 2026: Interconnection Queue Stress by Region', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'queue_by_region.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# trend: is the withdrawal rate worsening over time?
# group by year the project entered the queue, not when it resolved --
# this tells us whether more recent applicants are giving up more than older ones did
lbnl['q_year'] = lbnl['q_date'].dt.year
queue_trend = (
    lbnl
    .assign(withdrawn=lambda x: (x['q_status'] == 'withdrawn').astype(int))
    .groupby('q_year')
    .agg(projects=('q_id','count'), wd_rate=('withdrawn','mean'))
    .reset_index()
    .dropna()
)
queue_trend = queue_trend[(queue_trend['q_year'] >= 2005) & (queue_trend['projects'] > 50)]

fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()
ax1.bar(queue_trend['q_year'], queue_trend['projects'], color='lightsteelblue', alpha=0.6, label='Projects entering queue')
ax2.plot(queue_trend['q_year'], queue_trend['wd_rate'], 'o-', color='red', label='Withdrawal rate')
ax1.set_xlabel('Year project entered queue')
ax1.set_ylabel('Number of projects')
ax2.set_ylabel('Withdrawal rate'); ax2.set_ylim(0, 1)
ax1.set_title('US Interconnection Queue Volume and Withdrawal Rate Over Time')
lines = ax1.get_legend_handles_labels()[0] + ax2.get_legend_handles_labels()[0]
labels = ax1.get_legend_handles_labels()[1] + ax2.get_legend_handles_labels()[1]
ax1.legend(lines, labels, loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'queue_withdrawal_trend.png'), dpi=150, bbox_inches='tight')
plt.show()

## Section 11: Summary -- Investment Recommendation

Synthesis table combining all analysis outputs into a consulting-style recommendation.
Format mirrors what a Wood Mackenzie or ICF report would deliver.

In [ ]:
summary = top_stressed[['ba','region','congestion_stress_score','avg_renewable_share']].copy()

anomaly_rate_ba = df_anomaly.groupby('ba')['anomaly_flag'].mean().reset_index(name='anomaly_rate_pct')
summary = summary.merge(anomaly_rate_ba, on='ba', how='left')
summary = summary.merge(battery_df[['ba','annual_value_usd']], on='ba', how='left')

summary['avg_renewable_share']    = (summary['avg_renewable_share'] * 100).round(1).astype(str) + '%'
summary['anomaly_rate_pct']       = (summary['anomaly_rate_pct'] * 100).round(2).astype(str) + '%'
summary['annual_value_usd']       = summary['annual_value_usd'].apply(lambda x: f'${x/1e6:.2f}M' if not pd.isna(x) else 'N/A')
summary['congestion_stress_score'] = summary['congestion_stress_score'].round(3)
summary.columns = ['BA','Region','Stress Score','Renewable Share','Anomaly Rate','Est. Annual Battery Value']

print('=== INVESTMENT RECOMMENDATION SUMMARY ===')
print(f'Period: 2022-2024 | Battery: {BATTERY_MW}MW/{BATTERY_DURATION}-hr | ${AVOIDED_COST_PER_MWH}/MWh avoidance value')
summary

In [ ]:
print('Analysis complete.')
print(f'All output files saved to: {BASE_DIR}')
for f in sorted(os.listdir(BASE_DIR)):
    if f.endswith(('.png','.html')):
        print(f'  {f}')

## Section 12: Advanced GIS Maps

Four additional maps that go beyond the basic stress-score choropleth:

1. **Transmission Corridor Stress** -- every physical line segment colored by the stress score of its nearest balancing authority; you can see which corridors are structurally under pressure
2. **Net Interchange Flow Map** -- directional visualization showing which BAs are chronic exporters vs. importers, circle size scaled by magnitude; the power-highway view
3. **Renewable Generation Density (KDE)** -- kernel density heatmap weighted by average MW output, showing where solar+wind is geographically concentrated relative to transmission capacity
4. **Seasonal Stress Animation** -- GIF cycling through Spring/Summer/Fall/Winter so you can see how congestion bottlenecks shift with the seasons

All four reuse data already in memory from prior sections.

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.stats import gaussian_kde
from shapely.ops import nearest_points

# additional imports needed for the four advanced maps
# FuncAnimation/PillowWriter: animate the seasonal stress GIF
# gaussian_kde: kernel density estimation for the renewable heatmap
# nearest_points: spatial nearest-neighbor lookup for corridor stress coloring
print('Additional imports OK.')

In [ ]:
# ── MAP 1: TRANSMISSION CORRIDOR STRESS ─────────────────────────────────────
# what: color every HIFLD transmission line segment by the congestion stress score
#       of the nearest balancing authority centroid
# why: the basic map showed BA centroids as dots; this map paints the actual wires,
#      making it instantly clear which physical corridors are under structural pressure
# how: for a sample of line segments we find the nearest BA point (using shapely's
#      nearest_points on the unary union of BA centroids), inherit that BA's stress score,
#      bin scores into 8 quantile buckets, then plot each bucket in one geopandas call
#      (avoids iterating geometry row-by-row and handles MultiLineString correctly)

ba_points = gdf_conus[['ba','congestion_stress_score','geometry']].copy()

# every 5th line keeps rendering fast while still showing the full network structure
sample = transmission_conus.iloc[::5].copy()
sample['centroid'] = sample.geometry.centroid

# build spatial union of BA centroids for O(n log n) nearest-neighbor lookup
ba_geom_union = ba_points.geometry.unary_union

stress_scores = []
for centroid in sample['centroid']:
    nearest = nearest_points(centroid, ba_geom_union)[1]
    match = ba_points[ba_points.geometry == nearest]
    stress_scores.append(match.iloc[0]['congestion_stress_score'] if len(match) > 0 else np.nan)
sample['stress'] = stress_scores

fig, ax = plt.subplots(figsize=(18, 10))
us_states_conus.plot(ax=ax, color='#f0f0f0', edgecolor='#999999', linewidth=0.4)

vmin_c = sample['stress'].quantile(0.05)
vmax_c = sample['stress'].quantile(0.95)
norm_c = mcolors.Normalize(vmin=vmin_c, vmax=vmax_c)
cmap_c = plt.cm.RdYlGn_r

# bin stress into 8 quantile buckets; plot each bucket as one geopandas call
sample_valid = sample.dropna(subset=['stress']).copy()
sample_valid['stress_bin'] = pd.qcut(sample_valid['stress'], q=8, labels=False, duplicates='drop')
for bin_val, group in sample_valid.groupby('stress_bin'):
    mid_stress = group['stress'].median()
    color = cmap_c(norm_c(mid_stress))
    gpd.GeoDataFrame(group, geometry='geometry').plot(ax=ax, color=color, linewidth=0.6, alpha=0.7)

sm_c = plt.cm.ScalarMappable(cmap=cmap_c, norm=norm_c)
sm_c.set_array([])
plt.colorbar(sm_c, ax=ax, label='Congestion Stress Score (inherited from nearest BA)', shrink=0.6)

ax.set_xlim(-130, -60); ax.set_ylim(22, 52)
ax.set_title(
    'US Transmission Corridor Stress\n'
    'Line color = stress score of nearest balancing authority (red = most stressed)',
    fontsize=13
)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'map_corridor_stress.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Map 1 saved: map_corridor_stress.png')

In [ ]:
# ── MAP 2: NET INTERCHANGE FLOW MAP ─────────────────────────────────────────
# what: show each BA as a circle colored red (net exporter) or blue (net importer),
#       sized by the magnitude of its average net interchange; also saved as interactive HTML
# why: interchange flow reveals the structural power-highway pattern of the US grid --
#      which BAs are routinely pushing surplus power to others (and therefore most at risk
#      of curtailment when export capacity is saturated)
# how: avg_interchange_mw > 0 means the BA is a net importer (receives more than it sends);
#      < 0 means net exporter; we use TwoSlopeNorm centered at zero so the color scale
#      diverges symmetrically around the neutral point regardless of min/max asymmetry

interchange_ba = (
    df.groupby('ba')
    .agg(avg_interchange_mw=('interchange_mw','mean'), region=('region','first'))
    .reset_index()
)
interchange_ba['lat'] = interchange_ba['ba'].map(lambda x: ba_coords.get(x,(None,None))[0])
interchange_ba['lon'] = interchange_ba['ba'].map(lambda x: ba_coords.get(x,(None,None))[1])
interchange_ba = interchange_ba.dropna(subset=['lat','lon','avg_interchange_mw'])

fig, ax = plt.subplots(figsize=(18, 10))
us_states_conus.plot(ax=ax, color='#f0f0f0', edgecolor='#aaaaaa', linewidth=0.4)
transmission_conus.iloc[::10].plot(ax=ax, color='#dddddd', linewidth=0.3, alpha=0.5)

vmin_ic = interchange_ba['avg_interchange_mw'].quantile(0.05)
vmax_ic = interchange_ba['avg_interchange_mw'].quantile(0.95)
norm_ic = mcolors.TwoSlopeNorm(vcenter=0, vmin=vmin_ic, vmax=vmax_ic)
cmap_ic = plt.cm.RdBu  # blue = importer, red = exporter

max_mag = interchange_ba['avg_interchange_mw'].abs().max()
max_arrow = 3.0  # max arrow length in degrees

for _, row in interchange_ba.iterrows():
    mag   = row['avg_interchange_mw']
    scale = abs(mag) / max_mag * max_arrow
    dx = scale * np.sign(mag) * 0.6
    dy = scale * np.sign(mag) * 0.3
    color = cmap_ic(norm_ic(mag))
    ax.annotate(
        '', xy=(row['lon']+dx, row['lat']+dy), xytext=(row['lon'], row['lat']),
        arrowprops=dict(arrowstyle='->', color=color, lw=1.5 + abs(mag)/max_mag*2)
    )
    ax.scatter(row['lon'], row['lat'], s=30, color=color, zorder=5, edgecolor='black', linewidth=0.3)
    ax.annotate(row['ba'], (row['lon'], row['lat']), fontsize=5.5, ha='center', va='bottom',
                xytext=(0,4), textcoords='offset points', color='#333333')

sm_ic = plt.cm.ScalarMappable(cmap=cmap_ic, norm=norm_ic)
sm_ic.set_array([])
plt.colorbar(sm_ic, ax=ax, label='Avg Net Interchange (MW): positive = importing, negative = exporting', shrink=0.6)
ax.set_xlim(-130, -60); ax.set_ylim(22, 52)
ax.set_title(
    'US Grid Net Interchange Flows by Balancing Authority (2022-2024 Average)\n'
    'Arrow direction = import/export | Arrow size and color = magnitude',
    fontsize=13
)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'map_interchange_flows.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Map 2 (static) saved: map_interchange_flows.png')

# interactive folium version -- hover over each circle for exact MW values
m2 = folium.Map(location=[38,-96], zoom_start=4, tiles='CartoDB positron')
folium.GeoJson(
    transmission_conus.iloc[::10].__geo_interface__,
    style_function=lambda x: {'color':'#cccccc','weight':0.4,'opacity':0.5}
).add_to(m2)

for _, row in interchange_ba.iterrows():
    mag = row['avg_interchange_mw']
    is_export = mag < 0
    circle_color = '#d73027' if is_export else '#4575b4'
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=min(20, max(5, abs(mag)/500)),
        color='black', weight=0.5,
        fill=True, fill_color=circle_color, fill_opacity=0.75,
        tooltip=folium.Tooltip(
            f"<b>{row['ba']}</b> ({row['region']})<br>"
            f"Avg Net Interchange: {mag:+,.0f} MW<br>"
            f"{'Net EXPORTER (sends power out)' if is_export else 'Net IMPORTER (receives power)'}"
        )
    ).add_to(m2)

legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;
            padding:10px;border-radius:5px;border:1px solid #ccc;font-size:12px;">
  <b>Net Interchange (2022-2024 avg)</b><br>
  <span style="color:#d73027;">&#9679;</span> Net Exporter (sends power out)<br>
  <span style="color:#4575b4;">&#9679;</span> Net Importer (receives power in)<br>
  Circle size = magnitude (MW)
</div>"""
m2.get_root().html.add_child(folium.Element(legend_html))
m2.save(os.path.join(BASE_DIR, 'map_interchange_flows_interactive.html'))
print('Map 2 (interactive) saved: map_interchange_flows_interactive.html')

In [ ]:
# ── MAP 3: RENEWABLE GENERATION DENSITY (KDE HEATMAP) ───────────────────────
# what: kernel density estimation heatmap showing where solar+wind output is
#       geographically concentrated across the US, overlaid on transmission lines
# why: the KDE reveals mismatches between where renewables generate (geographic clusters)
#      and where transmission capacity exists -- hotspots with thin gray lines behind them
#      are the most likely curtailment candidates
# how: build a weighted point cloud by repeating each BA coordinate in proportion to its
#      average renewable output (more MW = more repeated points), then run scipy's
#      gaussian_kde on that cloud and evaluate it on a 300x200 CONUS grid;
#      the result is a smooth density surface that can be drawn as filled contours

ba_full = ba_profile.copy()
ba_full['lat'] = ba_full['ba'].map(lambda x: ba_coords.get(x,(None,None))[0])
ba_full['lon'] = ba_full['ba'].map(lambda x: ba_coords.get(x,(None,None))[1])

ren_df = ba_full.dropna(subset=['lat','lon','avg_renewable_mw'])
ren_df = ren_df[ren_df['avg_renewable_mw'] > 0]

# weight the point cloud by MW so the KDE density is proportional to output, not just BA count
weights  = ren_df['avg_renewable_mw'] / ren_df['avg_renewable_mw'].sum()
repeat_n = (weights * 2000).astype(int).clip(lower=1)
lats_kde = np.repeat(ren_df['lat'].values, repeat_n.values)
lons_kde = np.repeat(ren_df['lon'].values, repeat_n.values)

kde = gaussian_kde(np.vstack([lons_kde, lats_kde]), bw_method=0.15)

grid_lon = np.linspace(-130, -60, 300)
grid_lat = np.linspace(22, 52, 200)
Lon, Lat = np.meshgrid(grid_lon, grid_lat)
Z = kde(np.vstack([Lon.ravel(), Lat.ravel()])).reshape(Lon.shape)

fig, ax = plt.subplots(figsize=(18, 10))
us_states_conus.plot(ax=ax, color='none', edgecolor='#888888', linewidth=0.6, zorder=3)
cf = ax.contourf(Lon, Lat, Z, levels=20, cmap='YlOrRd', alpha=0.75, zorder=1)
plt.colorbar(cf, ax=ax, label='Renewable Generation Density (KDE, weighted by avg MW output)', shrink=0.6)
transmission_conus.iloc[::8].plot(ax=ax, color='#555555', linewidth=0.25, alpha=0.4, zorder=2)

# label the top 10 renewable BAs for reference
for _, row in ren_df.nlargest(10, 'avg_renewable_mw').iterrows():
    ax.annotate(
        f"{row['ba']}\n{row['avg_renewable_mw']:.0f} MW",
        xy=(row['lon'], row['lat']),
        xytext=(5, 5), textcoords='offset points',
        fontsize=7, fontweight='bold', color='#1a1a1a', zorder=5,
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none')
    )

ax.set_xlim(-130, -60); ax.set_ylim(22, 52)
ax.set_title(
    'US Renewable Generation Density Heatmap (2022-2024)\n'
    'Hotspots = where solar+wind output is geographically concentrated | Gray lines = transmission network',
    fontsize=13
)
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'map_renewable_kde.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Map 3 saved: map_renewable_kde.png')

In [ ]:
# ── MAP 4: SEASONAL STRESS ANIMATION ────────────────────────────────────────
# what: animated GIF cycling through Spring / Summer / Fall / Winter showing how
#       the congestion stress score of each BA shifts with the seasons
# why: a single annual-average map hides the most important story -- different parts
#      of the US grid are stressed at different times of year (PNM/SW in summer from
#      AC load peaks, NW hydro BAs stressed in late winter when snowpack is low,
#      wind-heavy Great Plains most stressed in spring when wind peaks and load is low)
# how: recompute the 3-feature stress score separately for each season's subset of hours;
#      use a global color scale (fixed vmin/vmax) so colors are comparable across frames;
#      FuncAnimation cycles through the 4 frames at 0.8 fps and PillowWriter exports to GIF

df['season'] = df['month'].map({
    12:'Winter', 1:'Winter', 2:'Winter',
    3:'Spring',  4:'Spring', 5:'Spring',
    6:'Summer',  7:'Summer', 8:'Summer',
    9:'Fall',   10:'Fall',  11:'Fall'
})

seasons = ['Spring', 'Summer', 'Fall', 'Winter']
season_stress = {}

for season in seasons:
    sub = df[df['season'] == season].copy()
    bp  = sub.groupby('ba').agg(
        avg_renewable_share = ('renewable_mw', lambda x: (x / sub.loc[x.index,'gen_total_mw'].replace(0,np.nan)).mean()),
        std_surplus         = ('gen_demand_surplus', 'std'),
        hours_total         = ('utc_time', 'count'),
    ).reset_index()
    heavy = sub[sub['gen_demand_surplus'] > sub['demand_mw']*0.5].groupby('ba').size().reset_index(name='h_heavy')
    bp    = bp.merge(heavy, on='ba', how='left')
    bp['heavy_export_pct'] = bp['h_heavy'] / bp['hours_total']
    sdf = bp[['ba','avg_renewable_share','std_surplus','heavy_export_pct']].dropna()
    sc  = StandardScaler().fit_transform(sdf[['avg_renewable_share','std_surplus','heavy_export_pct']])
    sdf = sdf.copy()
    sdf['stress'] = sc.mean(axis=1)
    sdf['lat']    = sdf['ba'].map(lambda x: ba_coords.get(x,(None,None))[0])
    sdf['lon']    = sdf['ba'].map(lambda x: ba_coords.get(x,(None,None))[1])
    season_stress[season] = sdf.dropna(subset=['lat','lon'])

# fixed color scale so frames are directly comparable
all_scores = np.concatenate([s['stress'].values for s in season_stress.values()])
vmin_s, vmax_s = np.percentile(all_scores, 5), np.percentile(all_scores, 95)
norm_s = mcolors.Normalize(vmin=vmin_s, vmax=vmax_s)
cmap_s = plt.cm.RdYlGn_r

season_colors = {'Spring':'#4dac26','Summer':'#d73027','Fall':'#f4a582','Winter':'#4575b4'}

fig2, ax2 = plt.subplots(figsize=(16, 9))
plt.subplots_adjust(left=0.05, right=0.88, top=0.92, bottom=0.06)
sm_s = plt.cm.ScalarMappable(cmap=cmap_s, norm=norm_s)
sm_s.set_array([])
fig2.colorbar(sm_s, ax=ax2, label='Congestion Stress Score', shrink=0.7, pad=0.02)

def draw_season(season_idx):
    ax2.clear()
    season = seasons[season_idx]
    sdf    = season_stress[season]

    us_states_conus.plot(ax=ax2, color='#f0f0f0', edgecolor='#aaaaaa', linewidth=0.5)
    transmission_conus.iloc[::10].plot(ax=ax2, color='#cccccc', linewidth=0.3, alpha=0.5)

    for _, row in sdf.iterrows():
        c = cmap_s(norm_s(row['stress']))
        ax2.scatter(row['lon'], row['lat'], s=120, color=c, edgecolor='black',
                    linewidth=0.5, zorder=5, alpha=0.9)

    for _, row in sdf.nlargest(5, 'stress').iterrows():
        ax2.annotate(row['ba'], (row['lon'], row['lat']),
                     xytext=(5,5), textcoords='offset points',
                     fontsize=7, fontweight='bold', color='darkred', zorder=6)

    ax2.set_xlim(-130, -60); ax2.set_ylim(22, 52)
    ax2.set_xlabel('Longitude'); ax2.set_ylabel('Latitude')
    ax2.set_title(
        'US Grid Congestion Stress by Season\n'
        f'Season: {season}   |   Circle color = stress score (red = most stressed)',
        fontsize=13, pad=8
    )

    # season badge strip at the bottom of the map area
    for i, s in enumerate(seasons):
        rect = mpatches.FancyBboxPatch(
            (-129 + i*18, 22.5), 15, 2.5,
            boxstyle='round,pad=0.3',
            facecolor=season_colors[s] if s == season else '#e0e0e0',
            edgecolor='#555555', linewidth=0.8, zorder=6
        )
        ax2.add_patch(rect)
        ax2.text(-129 + i*18 + 7.5, 23.75, s,
                 ha='center', va='center',
                 fontsize=9, fontweight='bold' if s == season else 'normal',
                 color='white' if s == season else '#666666', zorder=7)

anim = FuncAnimation(fig2, draw_season, frames=len(seasons), interval=1200, repeat=True)
gif_path = os.path.join(BASE_DIR, 'map_seasonal_stress.gif')
anim.save(gif_path, writer=PillowWriter(fps=0.8))
plt.close(fig2)
print(f'Map 4 saved: map_seasonal_stress.gif')
print(f'Open in any browser or image viewer to see the animation.')
print('\nAll 4 advanced GIS maps complete.')
for f in ['map_corridor_stress.png','map_interchange_flows.png','map_interchange_flows_interactive.html',
          'map_renewable_kde.png','map_seasonal_stress.gif']:
    path = os.path.join(BASE_DIR, f)
    size = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    print(f'  {f} ({size:.1f} MB)')